# Klasifikasi Churn Pelanggan E-commerce

## Tujuan
Membangun model klasifikasi biner untuk memperkirakan apakah pelanggan akan churn (`Churn = 1`) atau tidak (`Churn = 0`). Hasil prediksi dapat membantu perusahaan memprioritaskan pelanggan yang perlu ditawari promo atau dihubungi.

Notebook ini menggunakan `data_ecommerce_customer_churn_cleaned.csv`. Jalankan `python clean_customer_data.py` jika ingin membuat ulang file tersebut dari data mentah.

**Feature engineering:** notebook menambahkan `Inactive7Plus` (pesanan terakhir setidaknya 7 hari lalu) dan `DissatisfiedComplainant` (pelanggan mengeluh dengan skor kepuasan maksimal 2). **Feature selection:** `SelectFromModel` berbasis Random Forest memilih fitur setelah preprocessing dan hanya dipelajari dari data train.

Data dibagi secara stratified menjadi train (60%), validation (20%), dan test (20%). Validation digunakan untuk memilih model dan threshold berdasarkan F2 (yang memberi bobot lebih besar pada recall churn). Test hanya digunakan sekali untuk evaluasi akhir.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from data_preprocessing import (
    CATEGORICAL_FEATURES,
    FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    normalize_customer_columns,
    prepare_customer_features,
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Memuat dan memeriksa data

In [ ]:
DATA_PATH = Path("data_ecommerce_customer_churn_cleaned.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATA_PATH.resolve()}. "
        "Letakkan CSV di folder kerja notebook atau sesuaikan DATA_PATH."
    )

source_df = normalize_customer_columns(pd.read_csv(DATA_PATH))
target = TARGET
if target not in source_df.columns:
    raise ValueError(f"Kolom target tidak ditemukan: {target}")
target_values = pd.to_numeric(source_df[target], errors="raise")
if target_values.isna().any() or not set(target_values.unique()).issubset({0, 1}):
    raise ValueError("Target Churn harus berisi label biner 0 dan 1.")

X = prepare_customer_features(source_df)
y = target_values.astype(int)
df = X.copy()
df[target] = y

print(f"Ukuran data: {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"Duplikat: {df.duplicated().sum():,}")
display(df.head())
display(df.dtypes.rename("tipe data").to_frame())
display(df.isna().sum().rename("nilai kosong").to_frame())

## 2. Eksplorasi data
Periksa ketidakseimbangan target dan beberapa hubungan awal dengan churn. Grafik ini bersifat deskriptif, bukan bukti sebab-akibat.

In [ ]:
target_counts = df[target].value_counts().sort_index()
display(pd.DataFrame({"jumlah": target_counts, "proporsi": target_counts / len(df)}))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
target_counts.rename(index={0: "Tidak churn", 1: "Churn"}).plot(
    kind="bar", ax=axes[0], color=["#4C78A8", "#F58518"]
)
axes[0].set_title("Distribusi target")
axes[0].set_xlabel("Status pelanggan")
axes[0].set_ylabel("Jumlah")
axes[0].tick_params(axis="x", rotation=0)

churn_by_category = df.groupby("PreferedOrderCat", observed=False)[target].mean().sort_values()
churn_by_category.plot(kind="barh", ax=axes[1], color="#72B7B2")
axes[1].set_title("Proporsi churn per kategori pesanan")
axes[1].set_xlabel("Proporsi churn")
axes[1].set_ylabel("Kategori")

df.boxplot(column="Tenure", by=target, ax=axes[2], grid=False)
axes[2].set_title("Tenure berdasarkan status churn")
axes[2].set_xlabel("Churn (0 = tidak, 1 = ya)")
axes[2].set_ylabel("Tenure")
fig.suptitle("")
plt.tight_layout()
plt.show()

## 3. Membagi data
Target dijaga proporsinya pada tiap bagian dengan stratifikasi. Seluruh preprocessing dan seleksi fitur berada di dalam pipeline sehingga median imputasi, scaling, encoding, dan fitur terpilih dipelajari dari train saja.

In [ ]:
X = df[FEATURES]
y = df[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame(
    {
        "jumlah baris": [len(X_train), len(X_valid), len(X_test)],
        "proporsi churn": [y_train.mean(), y_valid.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
display(split_summary)

## 4. Preprocessing, feature selection, dan pelatihan model
Dua model pembanding digunakan: Logistic Regression sebagai baseline yang mudah ditafsirkan dan Random Forest untuk menangkap pola non-linear. Pipeline melakukan imputasi, scaling, one-hot encoding, lalu `SelectFromModel` dengan Random Forest (`threshold="median"`). Pembobotan kelas membantu menangani jumlah pelanggan churn yang lebih sedikit.

In [ ]:
def make_preprocessor():
    numeric_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_preprocessor, NUMERIC_FEATURES),
            ("categorical", categorical_preprocessor, CATEGORICAL_FEATURES),
        ]
    )


def make_feature_selector():
    return SelectFromModel(
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        threshold="median",
    )


models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}
pipelines = {}
for name, classifier in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor()),
            ("feature_selection", make_feature_selector()),
            ("classifier", classifier),
        ]
    )
    pipeline.fit(X_train, y_train)
    pipelines[name] = pipeline

print(f"Selesai melatih {len(pipelines)} model dengan seleksi fitur pada data train.")

## 5. Memilih model dan threshold dari validation set
Untuk setiap model, threshold dipilih dengan memaksimalkan F2 pada validation set. F2 mengutamakan recall, tetapi precision tetap diperhitungkan. Threshold ini bukan angka universal: semakin rendah threshold, semakin banyak pelanggan yang akan ditandai sebagai berisiko.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
validation_results = []
best_thresholds = {}

for name, pipeline in pipelines.items():
    valid_probabilities = pipeline.predict_proba(X_valid)[:, 1]
    threshold_scores = [
        fbeta_score(y_valid, valid_probabilities >= threshold, beta=2, zero_division=0)
        for threshold in thresholds
    ]
    best_index = int(np.argmax(threshold_scores))
    best_threshold = float(thresholds[best_index])
    best_thresholds[name] = best_threshold
    valid_predictions = valid_probabilities >= best_threshold
    validation_results.append(
        {
            "model": name,
            "threshold": best_threshold,
            "precision": precision_score(y_valid, valid_predictions, zero_division=0),
            "recall": recall_score(y_valid, valid_predictions, zero_division=0),
            "F2": threshold_scores[best_index],
            "ROC-AUC": roc_auc_score(y_valid, valid_probabilities),
            "PR-AUC (average precision)": average_precision_score(y_valid, valid_probabilities),
        }
    )

validation_results = pd.DataFrame(validation_results).sort_values(
    ["F2", "PR-AUC (average precision)"], ascending=False
).reset_index(drop=True)
display(validation_results)

best_model_name = validation_results.loc[0, "model"]
best_threshold = best_thresholds[best_model_name]
best_model = pipelines[best_model_name]
print(f"Model terpilih: {best_model_name}")
print(f"Threshold terpilih dari validation: {best_threshold:.2f}")
selected_names = best_model.named_steps["preprocessor"].get_feature_names_out()
selected_mask = best_model.named_steps["feature_selection"].get_support()
print("Fitur terpilih:", selected_names[selected_mask].tolist())

## 6. Evaluasi akhir pada test set
Model dan threshold sudah ditentukan tanpa melihat label test. Metrik test berikut merupakan estimasi performa pada data yang belum digunakan saat pemilihan.

In [ ]:
test_probabilities = best_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

test_metrics = pd.Series(
    {
        "precision": precision_score(y_test, test_predictions, zero_division=0),
        "recall": recall_score(y_test, test_predictions, zero_division=0),
        "F1": fbeta_score(y_test, test_predictions, beta=1, zero_division=0),
        "F2": fbeta_score(y_test, test_predictions, beta=2, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, test_probabilities),
        "PR-AUC (average precision)": average_precision_score(y_test, test_probabilities),
    },
    name="nilai test",
)
display(test_metrics.to_frame())
print("\nClassification report (label 1 = churn):")
print(classification_report(y_test, test_predictions, target_names=["Tidak churn", "Churn"], zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["Tidak churn", "Churn"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title(f"Confusion matrix — threshold {best_threshold:.2f}")
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[1], name=best_model_name)
axes[1].set_title("ROC curve")

precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probabilities)
axes[2].plot(recall_curve, precision_curve, color="#F58518")
axes[2].set_title("Precision-recall curve")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Interpretasi dan keterbatasan
- **Recall churn** menunjukkan proporsi pelanggan churn yang berhasil terdeteksi. Nilai lebih tinggi dapat membantu mengurangi pelanggan berisiko yang terlewat.
- **Precision churn** menunjukkan proporsi pelanggan yang ditandai berisiko dan benar-benar churn. Precision yang rendah dapat membuat biaya promo terbuang pada pelanggan yang tidak akan churn.
- **F2** merangkum precision dan recall dengan penekanan pada recall. Gunakan hasil validasi untuk pemilihan dan hasil test sebagai estimasi akhir yang lebih objektif.
- Threshold operasional sebaiknya ditetapkan bersama tim bisnis berdasarkan kapasitas kontak, biaya promo, dan dampak false positive/false negative. F2 saja belum memasukkan nilai ekonomi tersebut.
- Data historis dan fitur yang tersedia membatasi generalisasi model. Hubungan fitur dengan churn tidak membuktikan sebab-akibat; lakukan pemantauan, validasi berkala, dan pengujian dampak promo sebelum mengandalkan prediksi untuk keputusan pelanggan.